# Notebook 03 – Phân cụm ngày kiểu thời tiết (Clustering)

**Kỹ thuật:** KMeans + HAC (Hierarchical Agglomerative Clustering)  
**Lý do chọn KMeans:** Phổ biến, hiệu quả cho dữ liệu số; chọn K bằng Elbow + Silhouette + DBI.  
**Lý do so sánh HAC:** Kiểm tra tính ổn định kết quả – nếu HAC (Ward) ra kết quả tương tự → cụm robust.  

**Pipeline:**
1. Chuẩn hoá (StandardScaler) 7 đặc trưng số
2. Chọn K tối ưu (Elbow, Silhouette, Davies-Bouldin)
3. Fit KMeans + HAC → so sánh
4. Profiling cụm: đặt tên, phân tích đặc trưng
5. Giảm chiều PCA → scatter 2D

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.decomposition import PCA
from src.data.loader import load_config, load_processed_data
from src.features.builder import FeatureBuilder
from src.mining.clustering import WeatherClusterer
from src.evaluation.metrics import clustering_metrics
from src.evaluation.report import save_table
from src.visualization import plots

cfg = load_config('../configs/params.yaml')
plots.FIG_DIR = '../outputs/figures'
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

In [2]:
df = load_processed_data('../data/processed/weather_processed.parquet')
builder = FeatureBuilder(cfg)
X_scaled, feat_cols = builder.build_scaled_features(df)
print(f'Feature matrix: {X_scaled.shape}')
print(f'Features: {feat_cols}')

[loader] Loaded processed data: (95150, 16)
Feature matrix: (95150, 7)
Features: ['Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)']


In [3]:
# Lấy mẫu 20,000 điểm để tăng tốc
from sklearn.utils import resample
idx = resample(range(len(df)), n_samples=min(20000, len(df)), random_state=42)
X_sample = X_scaled[idx]
df_sample = df.iloc[idx].reset_index(drop=True)
print(f'Sample size: {X_sample.shape}')

Sample size: (20000, 7)


## 3.1 Chọn K tối ưu

In [4]:
clusterer = WeatherClusterer(cfg)
k_results = clusterer.find_best_k(X_sample)
print(k_results.to_string(index=False))
save_table(k_results, 'cluster_k_selection', '../outputs/tables')

  k=2: silhouette=0.2457, DBI=1.5314


  k=3: silhouette=0.1939, DBI=1.6715


  k=4: silhouette=0.1977, DBI=1.6193


  k=5: silhouette=0.1913, DBI=1.5394


  k=6: silhouette=0.1802, DBI=1.5447


  k=7: silhouette=0.1794, DBI=1.4634


  k=8: silhouette=0.1668, DBI=1.5469
 k       inertia  silhouette    dbi
 2 101212.721914      0.2457 1.5314
 3  86584.388292      0.1939 1.6715
 4  76493.778923      0.1977 1.6193
 5  70075.904935      0.1913 1.5394
 6  64929.673740      0.1802 1.5447
 7  60551.869404      0.1794 1.4634
 8  57679.235131      0.1668 1.5469
[report] Saved table: ../outputs/tables\cluster_k_selection.csv


'../outputs/tables\\cluster_k_selection.csv'

In [5]:
plots.plot_elbow_silhouette(k_results)

# Chọn K tự động: silhouette cao nhất
best_row = k_results.loc[k_results['silhouette'].idxmax()]
print(f'K tối ưu theo Silhouette: k={int(best_row["k"])} (silhouette={best_row["silhouette"]}, DBI={best_row["dbi"]})')

[plots] Saved: ../outputs/figures\cluster_elbow_silhouette_dbi.png
K tối ưu theo Silhouette: k=2 (silhouette=0.2457, DBI=1.5314)


## 3.2 Fit KMeans & HAC → So sánh

In [6]:
labels_km = clusterer.fit_kmeans(X_sample, k=4)
km_metrics = clustering_metrics(X_sample, labels_km)
print('KMeans:', km_metrics)

labels_hac = clusterer.fit_hac(X_sample, k=4)
hac_metrics = clustering_metrics(X_sample, labels_hac)
print('HAC:', hac_metrics)

print(f'\nSo sánh:')
print(f'  KMeans silhouette={km_metrics["silhouette"]:.4f} vs HAC silhouette={hac_metrics["silhouette"]:.4f}')
better = 'KMeans' if km_metrics['silhouette'] >= hac_metrics['silhouette'] else 'HAC'
print(f'  → {better} có silhouette cao hơn')

[cluster] KMeans k=4: silhouette=0.1977, DBI=1.6193


KMeans: {'silhouette': 0.1977, 'davies_bouldin': 1.6193, 'n_clusters': 4}


[cluster] HAC k=4: silhouette=0.1304


HAC: {'silhouette': 0.1304, 'davies_bouldin': 2.0366, 'n_clusters': 4}

So sánh:
  KMeans silhouette=0.1977 vs HAC silhouette=0.1304
  → KMeans có silhouette cao hơn


## 3.3 Profiling cụm

In [7]:
profile = clusterer.profile_clusters(df_sample, labels_km, feat_cols)
print(profile.to_string())
save_table(profile, 'cluster_profile', '../outputs/tables')

         Temperature (C)  Apparent Temperature (C)  Humidity  Wind Speed (km/h)  Wind Bearing (degrees)  Visibility (km)  Pressure (millibars)  Count   Pct DominantWeather DominantSeason  ClusterName
Cluster                                                                                                                                                                                                
0                  7.139                     4.504     0.774             19.206                 217.984           10.811              1011.126   3660  18.3          Cloudy         Winter  Cold & Damp
1                 23.833                    23.795     0.467             11.977                 187.264           11.725              1015.407   4739  23.7          Cloudy         Summer    Hot & Dry
2                  1.667                    -0.316     0.880              7.901                 168.021            5.952              1023.221   5154  25.8          Cloudy         Winter  Arctic Cold


'../outputs/tables\\cluster_profile.csv'

In [8]:
plots.plot_cluster_profile(profile, feat_cols)

[plots] Saved: ../outputs/figures\cluster_profile_bar.png


## 3.4 PCA → Scatter 2D

In [9]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_sample)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}')
plots.plot_cluster_scatter(X_2d, labels_km, 'KMeans Clusters (PCA 2D)')

PCA variance explained: 0.589


[plots] Saved: ../outputs/figures\cluster_scatter_pca.png


## 3.5 Insight phân cụm (tự động từ profile)

In [10]:
print('=== INSIGHT PHÂN CỤM ===')
for idx, row in profile.iterrows():
    name = row.get('ClusterName', f'Cluster {idx}')
    temp = row.get('Temperature (C)', 0)
    humid = row.get('Humidity', 0)
    wind = row.get('Wind Speed (km/h)', 0)
    vis = row.get('Visibility (km)', 0)
    count = int(row.get('Count', 0))
    pct = row.get('Pct', 0)
    dom_w = row.get('DominantWeather', '?')
    dom_s = row.get('DominantSeason', '?')
    print(f'\n→ Cluster {idx} ({name}): {count:,} mẫu ({pct}%)')
    print(f'  Temp={temp:.1f}°C, Humidity={humid:.2f}, Wind={wind:.1f} km/h, Vis={vis:.1f} km')
    print(f'  Thời tiết chủ đạo: {dom_w} | Mùa chủ đạo: {dom_s}')

print('\nKhuyến nghị tự động:')
for idx, row in profile.iterrows():
    name = row.get('ClusterName', f'Cluster {idx}')
    temp = row.get('Temperature (C)', 10)
    vis = row.get('Visibility (km)', 10)
    if temp <= 2:
        print(f'  {name}: Cảnh báo đóng băng, nhiệt độ cực thấp')
    elif vis < 5:
        print(f'  {name}: Cảnh báo tầm nhìn thấp, nguy hiểm giao thông')
    elif temp >= 25:
        print(f'  {name}: Thích hợp hoạt động ngoài trời, du lịch')
    else:
        print(f'  {name}: Điều kiện ôn hoà')

=== INSIGHT PHÂN CỤM ===

→ Cluster 0 (Cold & Damp): 3,660 mẫu (18.3%)
  Temp=7.1°C, Humidity=0.77, Wind=19.2 km/h, Vis=10.8 km
  Thời tiết chủ đạo: Cloudy | Mùa chủ đạo: Winter

→ Cluster 1 (Hot & Dry): 4,739 mẫu (23.7%)
  Temp=23.8°C, Humidity=0.47, Wind=12.0 km/h, Vis=11.7 km
  Thời tiết chủ đạo: Cloudy | Mùa chủ đạo: Summer

→ Cluster 2 (Arctic Cold): 5,154 mẫu (25.8%)
  Temp=1.7°C, Humidity=0.88, Wind=7.9 km/h, Vis=6.0 km
  Thời tiết chủ đạo: Cloudy | Mùa chủ đạo: Winter

→ Cluster 3 (Mild): 6,447 mẫu (32.2%)
  Temp=14.0°C, Humidity=0.80, Wind=7.3 km/h, Vis=12.7 km
  Thời tiết chủ đạo: Cloudy | Mùa chủ đạo: Summer

Khuyến nghị tự động:
  Cold & Damp: Điều kiện ôn hoà
  Hot & Dry: Điều kiện ôn hoà
  Arctic Cold: Cảnh báo đóng băng, nhiệt độ cực thấp
  Mild: Điều kiện ôn hoà
